In [ ]:
import rosbag
from matplotlib import pyplot as plt
import numpy as np
import bagpy
from bagpy import bagreader

In [ ]:
def ReadBatteryVoltageAndFlyStatus(bag_path):
    """ Example bag file:
    path:        battery.bag
    version:     2.0
    duration:    8:25s (505s)
    start:       Jan 01 1970 00:03:57.05 (237.05)
    end:         Jan 01 1970 00:12:22.69 (742.69)
    size:        93.8 KB
    messages:    759
    compression: none [1/1 chunks]
    types:       mavros_msgs/State        [65cd0a9fff993b062b91e354554ec7e9]
                sensor_msgs/BatteryState [4ddae7f048e32fda22cac764685e3974]
    topics:      /uav2/mavros/battery   253 msgs    : sensor_msgs/BatteryState
                /uav2/mavros/state     506 msgs    : mavros_msgs/State
    """
    bag = rosbag.Bag(bag_path)
    battery_voltage = []
    battery_current = []
    armed = []
    t0 = None
    for topic, msg, t in bag.read_messages(topics=['/uav2/mavros/battery', '/uav2/mavros/state']):
        if t0 is None:
            t0 = t
        t1 = (msg.header.stamp - t0).to_sec()
        # Check if the message is of type sensor_msgs/BatteryState
        if msg._type == 'sensor_msgs/BatteryState':
            battery_voltage.append([t1, msg.voltage])
            battery_current.append([t1, -msg.current])
        # Check if the message is of type mavros_msgs/State
        if msg._type == 'mavros_msgs/State':
            armed.append([t1, msg.armed])
    bag.close()
    return np.array(battery_voltage), np.array(battery_current), np.array(armed)

    
battery_voltages, battery_current, armed = ReadBatteryVoltageAndFlyStatus("/home/xuhao/output/battery.bag")
# Plot the battery voltage and fly status
fig, ax1 = plt.subplots()
color = 'tab:red'
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Battery Voltage (V)', color=color)
ax1.plot(battery_voltages[:,0], battery_voltages[:,1], color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid()

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Battery Current (A)', color=color)
ax2.plot(battery_current[:,0], battery_current[:,1], color=color)
ax2.tick_params(axis='y', labelcolor=color)
ax2.grid()

